# SKR Optimization Grid Search

This notebook performs a grid search optimization for Secret Key Rate (SKR) in a BB84 server-client simulation, treating the mean photon number ($\mu$) as a central variable across different error correction protocols.

In [1]:
import asyncio
import os
import sys
import time
import itertools
import nest_asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

nest_asyncio.apply()

# Ensure we can import from the ServerClientBB84 directory
current_dir = os.getcwd()
target_dir = os.path.join(current_dir, 'ServerClientBB84')
if target_dir not in sys.path:
    sys.path.append(target_dir)

try:
    from bb84_server_client import AliceServer, BobClient, QuantumChannel, Detector, APIClient
except ImportError:
    sys.path.append(os.getcwd())
    from bb84_server_client import AliceServer, BobClient, QuantumChannel, Detector, APIClient

print("Libraries imported successfully.")

[NR-LDPC] GPU Detected: 1 device(s). Acceleration ENABLED.
[NR-LDPC] Sionna Library Loaded Successfully.
Libraries imported successfully.


## Define Base Physical Parameters and Optimization Grid

In [2]:
# Base Simulation Parameters
source_freq_hz = 1e7
block_size = int(50000)  # Reduced block size for faster grid search iteration
fiber_loss_db_km = 0.2
detector_efficiency = 0.8
dark_count_rate = 1e4
time_window = 1e-9
optical_error_rate = 0.01
dark_count_prob = 1 - np.exp(-dark_count_rate * time_window)

print('Common Physical Parameters:')
print(f'  Source frequency (Hz): {source_freq_hz:.2e}')
print(f'  Block size / emitted qubits: {block_size}')
print(f'  Fiber loss (dB/km): {fiber_loss_db_km}')
print(f'  Detector efficiency: {detector_efficiency}')
print(f'  Dark count probability: {dark_count_prob:.3e}')

# Grid Search Parameters
# Define the range of variables to sweep over
mu_values = np.linspace(0.001, 1, 20) 
distances_km = [20,30,40,50]
protocols = ['cascade', 'winnow', 'nr_ldpc_std', 'polar', 'ldpc_rateadaptive']

print(f'\nGrid Search Axes:')
print(f'  Mu values: {mu_values}')
print(f'  Distances: {distances_km}')
print(f'  Protocols: {protocols}')

Common Physical Parameters:
  Source frequency (Hz): 1.00e+07
  Block size / emitted qubits: 50000
  Fiber loss (dB/km): 0.2
  Detector efficiency: 0.8
  Dark count probability: 1.000e-05

Grid Search Axes:
  Mu values: [0.001      0.05357895 0.10615789 0.15873684 0.21131579 0.26389474
 0.31647368 0.36905263 0.42163158 0.47421053 0.52678947 0.57936842
 0.63194737 0.68452632 0.73710526 0.78968421 0.84226316 0.89484211
 0.94742105 1.        ]
  Distances: [20, 30, 40, 50]
  Protocols: ['cascade', 'winnow', 'nr_ldpc_std', 'polar', 'ldpc_rateadaptive']


## Define Asynchronous Evaluation Function

In [3]:
async def evaluate_skr(distance, mu, protocol):
    """
    Runs the OOP simulation for a given set of hyperparameters.
    """
    # Instantiate authentic Original objects (inspired by finite vs server client)
    channel = QuantumChannel("Fiber", distance, fiber_loss_db_km, optical_error_rate, next_actor=None)
    alice = AliceServer("Alice", channel, num_qubits=block_size, mu=mu, verbose=False)
    api = APIClient(alice)
    
    # We use Toeplitz PA and the respective iteration protocol
    bob = BobClient("Bob", api, protocol=protocol, pa_protocol="toeplitz", verbose=False)
    detector = Detector("Det", detector_efficiency, dark_count_prob, parent_bob=bob)
    
    # Complete Actor loop wiring
    channel.next_actor = detector
    
    # Start actors inside our loop
    actors = [alice, channel, detector, bob]
    tasks = [asyncio.create_task(a.start()) for a in actors]
    
    # 1. Quantum phase
    await alice.run_quantum_transmission()
    
    # Wait until all photon objects drain from queues
    while (not channel.mailbox.empty() or 
           not detector.mailbox.empty() or 
           not bob.mailbox.empty()):
        await asyncio.sleep(0.01)
    
    # Minute buffer for final detection handling
    await asyncio.sleep(0.05)
    
    # 2. Setup Classical Bounds dynamically
    T = channel.transmittance
    eta = detector.eta
    
    p_sig = 1.0 - np.exp(-mu * T * eta)
    p_dark = 2 * detector.p_dc
    p_click = p_sig + p_dark - (p_sig * p_dark)
    p_multi = 1.0 - (1.0 + mu) * np.exp(-mu)
    
    # PNS bound
    q1 = max(0.0, 1.0 - (p_multi / p_click)) if p_click > 0 else 0.0
    
    # 3. Post-Processing Phase directly via Bob object
    pa_length = 0
    try:
        results = await bob.run_classical_post_processing(alice.num_qubits, q1=q1)
        pa_length = results.get('pa_length', 0)
    except Exception as e:
        # Errors typically arise if Sifted key string is entirely exhausted due to extreme QBER
        pass
        
    # Teardown logic
    for a in actors:
        await a.send(a, ("STOP",))
    for t in tasks:
        t.cancel()
        
    skr = (pa_length / block_size) * source_freq_hz if pa_length > 0 else 0.0
    return skr

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coro)

## Execute Grid Search Optimization

In [4]:
results_list = []

print("Starting Grid Search...")
start_time = time.time()

# Generate all parameter combinations
param_combinations = list(itertools.product(distances_km, protocols, mu_values))
total_runs = len(param_combinations)

for i, (dist, proto, mu) in enumerate(param_combinations):
    print(f"[{i+1}/{total_runs}] Evaluating Distance={dist}km, Protocol={proto}, Mu={mu:.3f} ...", end="")
    
    t_start_iter = time.time()
    skr_val = run_async(evaluate_skr(dist, mu, proto))
    t_end_iter = time.time()
    
    print(f" SKR: {skr_val:.2e} bits/s (took {t_end_iter - t_start_iter:.2f}s)")
    
    results_list.append({
        'Distance': dist,
        'Protocol': proto,
        'Mu': mu,
        'SKR': skr_val
    })

print(f"\nGrid search completed in {time.time() - start_time:.2f} seconds.")

Starting Grid Search...
[1/400] Evaluating Distance=20km, Protocol=cascade, Mu=0.001 ...
[BOB] Processing 15 detection events.
[BOB] First 15 Valid Detections: 0(x), 0(+), 1(+), 0(+), 0(x), 1(+), 0(x), 0(+), 1(+), 0(x), 0(+), 1(+), 1(x), 0(x), 0(x) ...
[BOB] Sending 15 bases to Alice for sifting...
[BOB] Sifting complete. Bases matched on 7 events.
[BOB] Sifted Key Preview: [0, 1, 0, 0, 0, 0, 1]...
[BOB] Error: Sifted key too short (7) for QBER estimation.
 SKR: 0.00e+00 bits/s (took 0.67s)
[2/400] Evaluating Distance=20km, Protocol=cascade, Mu=0.054 ...
[BOB] Processing 797 detection events.
[BOB] First 15 Valid Detections: 1(+), 1(x), 0(x), 0(+), 1(+), 1(x), 1(+), 0(+), 1(+), 1(x), 0(x), 0(x), 1(+), 0(+), 1(x) ...
[BOB] Sending 797 bases to Alice for sifting...
[BOB] Sifting complete. Bases matched on 394 events.
[BOB] Sifted Key Preview: [1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1]...
[BOB] Sampling 78 bits for QBER estimation...
[BOB] QBER Analysis: 0 errors in 78 s

I0000 00:00:1779263995.120072   60608 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4748 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060 with Max-Q Design, pci bus id: 0000:01:00.0, compute capability: 7.5



[BOB] Starting Privacy Amplification using toeplitz...

      PRIVACY AMPLIFICATION METRICS
 [+] Reconciled Key Length (N):      338 bits
 [+] QBER (e):                       1.1905%
 [+] Est. Eve's Knowledge Bound (t): 100 bits
 [+] Security Parameter (S):         10 bits
 [+] Final Secret Key Length (R):    228 bits
[BOB] Sent PA seed and protocol choice to Alice. Generated secret key of length 228.
 SKR: 4.56e+04 bits/s (took 5.92s)
[43/400] Evaluating Distance=20km, Protocol=nr_ldpc_std, Mu=0.106 ...
[BOB] Processing 1680 detection events.
[BOB] First 15 Valid Detections: 1(+), 1(x), 1(+), 1(x), 0(x), 0(+), 0(+), 0(x), 1(x), 0(x), 0(x), 0(x), 1(+), 1(x), 1(+) ...
[BOB] Sending 1680 bases to Alice for sifting...
[BOB] Sifting complete. Bases matched on 831 events.
[BOB] Sifted Key Preview: [1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0]...
[BOB] Sampling 166 bits for QBER estimation...
[BOB] QBER Analysis: 1 errors in 166 samples. Estimated QBER = 0.60%
[BOB] Discardin

## Process and Analyze Results

In [5]:
df_results = pd.DataFrame(results_list)

# Find the optimal Mu for each Protocol and Distance combination
optimal_configs = df_results.loc[df_results.groupby(['Distance', 'Protocol'])['SKR'].idxmax()]

print("Optimal Configurations (Max SKR):")
display(optimal_configs.reset_index(drop=True))

Optimal Configurations (Max SKR):


,Distance,Protocol,Mu,SKR
0,20,cascade,0.421632,153000.0
1,20,ldpc_rateadaptive,0.263895,102600.0
2,20,nr_ldpc_std,0.369053,187600.0
3,20,polar,0.263895,148000.0
4,20,winnow,0.316474,161400.0
5,30,cascade,0.211316,65200.0
6,30,ldpc_rateadaptive,0.158737,11000.0
7,30,nr_ldpc_std,0.211316,54000.0
8,30,polar,0.211316,61400.0
9,30,winnow,0.263895,71800.0
